# BirdCLEF+ 2026 — Diagnostic Training V7
Simplified pipeline to debug AUC=0.50 issue

In [ ]:
import subprocess, sys, os

# 在 import torch 之前检测 GPU，确保安装正确的 PyTorch 版本
try:
    r = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                       capture_output=True, text=True, timeout=10)
    gpu_caps = [float(x.strip()) for x in r.stdout.strip().split('\n') if x.strip()]
    min_cap = min(gpu_caps) if gpu_caps else 99.0
    print(f'GPU compute capabilities: {gpu_caps}')
except Exception:
    min_cap = 99.0
    print('nvidia-smi failed, assuming compatible GPU')

if min_cap < 7.0:
    print(f'SM {min_cap} < 7.0 (P100), installing PyTorch 2.5.1+cu124 BEFORE import...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'torch==2.5.1', 'torchaudio==2.5.1',
        '--index-url', 'https://download.pytorch.org/whl/cu124'])
    print('Compatible PyTorch installed')

import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')

In [ ]:
import os, sys, time, ast, random, warnings, glob as globmod
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchaudio
import timm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

N_GPUS = torch.cuda.device_count()
print(f'PyTorch {torch.__version__}, torchaudio {torchaudio.__version__}, CUDA: {torch.cuda.is_available()}, GPUs: {N_GPUS}')
for i in range(N_GPUS):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

USE_AMP = torch.cuda.is_available()
try:
    from torch.amp import autocast, GradScaler
    AMP_DEVICE = 'cuda'
except ImportError:
    from torch.cuda.amp import autocast, GradScaler
    AMP_DEVICE = None

In [ ]:
def find_data_dir():
    if not Path('/kaggle/input').exists(): return Path('../data/raw')
    for p in Path('/kaggle/input').iterdir():
        if (p / 'train.csv').exists(): return p
        if p.is_dir():
            for sub in p.iterdir():
                if sub.is_dir() and (sub / 'train.csv').exists(): return sub
    raise FileNotFoundError('train.csv not found')

class CFG:
    IS_KAGGLE = Path('/kaggle/input').exists()
    DATA_DIR = find_data_dir()
    OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('../models')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    SR = 32_000; CLIP_DURATION = 5.0; CLIP_SAMPLES = int(SR * CLIP_DURATION)
    N_FFT = 1024; HOP_LENGTH = 320; N_MELS = 128; FMIN = 50; FMAX = 14000
    NUM_CLASSES = 234; EPOCHS = 10; BATCH_SIZE = 64
    LR = 1e-3; WEIGHT_DECAY = 1e-4; NUM_WORKERS = 4; SEED = 42
    N_FOLDS = 5; TRAIN_FOLDS = [0]
    SEC_LABEL_W = 0.3
    MIXUP_P = 0.3; MIXUP_ALPHA = 0.15
    TMASK_P = 0.3; TMASK_W = 50; FMASK_P = 0.3; FMASK_W = 20
    NOISE_P = 0.2; NOISE_STD = 0.005; GAIN_P = 0.3; GAIN_RANGE = (-6, 6)

def set_seed(seed=CFG.SEED):
    random.seed(seed); os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False; torch.backends.cudnn.benchmark = True

set_seed()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Data: {CFG.DATA_DIR}, Device: {DEVICE}')

In [ ]:
train_df = pd.read_csv(CFG.DATA_DIR / 'train.csv')
taxonomy = pd.read_csv(CFG.DATA_DIR / 'taxonomy.csv')
submission = pd.read_csv(CFG.DATA_DIR / 'sample_submission.csv', nrows=0)
SPECIES_COLS = [c for c in submission.columns if c != 'row_id']
LABEL2IDX = {label: i for i, label in enumerate(SPECIES_COLS)}
TAX_MAP = dict(zip(taxonomy['primary_label'].astype(str), taxonomy['class_name']))
print(f'Samples: {len(train_df)}, Species: {len(SPECIES_COLS)}')
print(f'Columns: {list(train_df.columns)}')
print(f'\nSample rows:')
print(train_df[['filename','primary_label','secondary_labels','author']].head(3))
print(f'\nAudio files example:')
audio_dir = CFG.DATA_DIR / 'train_audio'
sample_files = list(audio_dir.iterdir())[:5]
for f in sample_files:
    print(f'  {f.name} ({f.stat().st_size/1024:.1f}KB)')

In [ ]:
# ===== DIAGNOSTIC: Test audio loading =====
MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(
    sample_rate=CFG.SR, n_fft=CFG.N_FFT, hop_length=CFG.HOP_LENGTH,
    n_mels=CFG.N_MELS, f_min=CFG.FMIN, f_max=CFG.FMAX, power=2.0)
AMP_TO_DB = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=80)

RESAMPLERS = {}

def load_audio_fast(path, sr=CFG.SR, duration=CFG.CLIP_DURATION, offset=0.0):
    tl = int(sr * duration)
    try:
        waveform, file_sr = torchaudio.load(path)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if file_sr != sr:
            if file_sr not in RESAMPLERS:
                RESAMPLERS[file_sr] = torchaudio.transforms.Resample(file_sr, sr)
            waveform = RESAMPLERS[file_sr](waveform)
        start = int(offset * sr)
        waveform = waveform[:, start:start + tl].squeeze(0)
    except Exception as e:
        print(f'  LOAD FAILED: {path} -> {e}')
        waveform = torch.zeros(tl)
    if waveform.numel() == 0:
        waveform = torch.zeros(tl)
    if waveform.numel() < tl:
        reps = (tl // waveform.numel()) + 1
        waveform = waveform.repeat(reps)[:tl]
    return waveform[:tl]

def audio_to_melspec_fast(waveform):
    mel = AMP_TO_DB(MEL_TRANSFORM(waveform.unsqueeze(0))).squeeze(0)
    return (mel - mel.mean()) / (mel.std() + 1e-6)

# Test on 5 random files
print('=== Audio Loading Test ===')
test_rows = train_df.sample(5, random_state=42)
for _, row in test_rows.iterrows():
    fpath = str(audio_dir / row['filename'])
    audio = load_audio_fast(fpath)
    mel = audio_to_melspec_fast(audio)
    print(f'  {row["filename"]}: audio={audio.shape} min={audio.min():.4f} max={audio.max():.4f} std={audio.std():.4f} | mel={mel.shape} min={mel.min():.2f} max={mel.max():.2f} std={mel.std():.2f}')
    is_silence = audio.abs().max() < 1e-6
    if is_silence:
        print(f'    WARNING: Audio is silence!')

In [ ]:
class BirdDataset(Dataset):
    def __init__(self, df, is_train=True):
        self.df = df.reset_index(drop=True)
        self.is_train = is_train
        self.audio_dir = CFG.DATA_DIR / 'train_audio'

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        offset = 0.0
        if self.is_train:
            offset = np.random.uniform(0, 25.0)  # random offset in 30s clip

        audio = load_audio_fast(str(self.audio_dir / row['filename']), offset=offset)

        if self.is_train:
            if np.random.rand() < CFG.NOISE_P:
                audio = audio + torch.randn_like(audio) * CFG.NOISE_STD
            if np.random.rand() < CFG.GAIN_P:
                audio = audio * (10 ** (np.random.uniform(*CFG.GAIN_RANGE) / 20))

        mel = audio_to_melspec_fast(audio)

        if self.is_train:
            nm, nf = mel.shape
            if np.random.rand() < CFG.TMASK_P:
                tw = np.random.randint(0, min(CFG.TMASK_W, nf))
                t0 = np.random.randint(0, max(1, nf - tw))
                mel[:, t0:t0+tw] = 0
            if np.random.rand() < CFG.FMASK_P:
                fw = np.random.randint(0, min(CFG.FMASK_W, nm))
                f0 = np.random.randint(0, max(1, nm - fw))
                mel[f0:f0+fw, :] = 0

        target = torch.zeros(CFG.NUM_CLASSES, dtype=torch.float32)
        p = str(row['primary_label'])
        if p in LABEL2IDX: target[LABEL2IDX[p]] = 1.0
        sr = row.get('secondary_labels', '[]')
        if isinstance(sr, str) and sr not in ('[]', ''):
            try:
                for s in ast.literal_eval(sr):
                    if str(s) in LABEL2IDX: target[LABEL2IDX[str(s)]] = CFG.SEC_LABEL_W
            except: pass

        return mel.unsqueeze(0), target

def mixup(mel, tgt, alpha=CFG.MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha)
    p = torch.randperm(mel.size(0))
    return lam*mel+(1-lam)*mel[p], lam*tgt+(1-lam)*tgt[p]

In [ ]:
class BirdCLEFB0(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model('tf_efficientnet_b0_ns', pretrained=pretrained,
                                          in_chans=1, num_classes=0, global_pool='avg')
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(1280, CFG.NUM_CLASSES)
    def forward(self, mel):
        f = self.backbone(mel)
        return self.fc(self.dropout(f))

_m = BirdCLEFB0(pretrained=False)
_out = _m(torch.randn(2, 1, 128, 500))
print(f'Model output: {_out.shape}, {sum(p.numel() for p in _m.parameters())/1e6:.1f}M params')
print(f'Output stats: mean={_out.mean():.4f}, std={_out.std():.4f}, min={_out.min():.4f}, max={_out.max():.4f}')
del _m, _out

In [ ]:
def _amp():
    return autocast(AMP_DEVICE) if AMP_DEVICE else autocast()

def train_ep(model, loader, crit, opt, scaler, ep_num=0):
    model.train(); tl, n = 0.0, 0
    for bi, (mel, tgt) in enumerate(tqdm(loader, leave=False)):
        mel, tgt = mel.to(DEVICE, non_blocking=True), tgt.to(DEVICE, non_blocking=True)
        if np.random.rand() < CFG.MIXUP_P: mel, tgt = mixup(mel, tgt)
        opt.zero_grad(set_to_none=True)
        if scaler:
            with _amp():
                logits = model(mel)
                loss = crit(logits, tgt)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        else:
            logits = model(mel)
            loss = crit(logits, tgt); loss.backward(); opt.step()

        # Diagnostic: print first batch stats for epoch 0
        if ep_num == 0 and bi == 0:
            with torch.no_grad():
                probs = torch.sigmoid(logits)
                print(f'\n  [DIAG] First batch:')
                print(f'    mel: shape={mel.shape} mean={mel.mean():.4f} std={mel.std():.4f}')
                print(f'    target: sum_per_sample={tgt.sum(dim=1).mean():.2f} (expect ~1.0-1.3)')
                print(f'    logits: mean={logits.mean():.4f} std={logits.std():.4f} min={logits.min():.4f} max={logits.max():.4f}')
                print(f'    probs:  mean={probs.mean():.4f} std={probs.std():.4f} min={probs.min():.4f} max={probs.max():.4f}')
                print(f'    loss: {loss.item():.4f}')
                pos_mask = tgt > 0.5
                if pos_mask.any():
                    print(f'    prob@pos: mean={probs[pos_mask].mean():.4f} std={probs[pos_mask].std():.4f}')
                neg_mask = tgt < 0.1
                if neg_mask.any():
                    print(f'    prob@neg: mean={probs[neg_mask].mean():.4f} std={probs[neg_mask].std():.4f}')

        tl += loss.item()*mel.size(0); n += mel.size(0)
    return tl/n

@torch.no_grad()
def val(model, loader, ep_num=0):
    model.eval(); ps, ts = [], []
    for bi, (mel, tgt) in enumerate(tqdm(loader, leave=False)):
        mel = mel.to(DEVICE, non_blocking=True)
        if USE_AMP:
            with _amp(): logits = model(mel)
        else: logits = model(mel)
        probs = torch.sigmoid(logits).cpu().numpy()
        ps.append(probs); ts.append(tgt.numpy())

        if ep_num == 0 and bi == 0:
            print(f'  [DIAG] Val first batch: logits mean={logits.mean():.4f} std={logits.std():.4f}')
            print(f'    probs mean={probs.mean():.4f} std={probs.std():.4f} min={probs.min():.4f} max={probs.max():.4f}')

    ap, at = np.concatenate(ps), np.concatenate(ts)
    at_bin = (at > 0.5).astype(np.float32)

    # Diagnostic: check prediction distribution
    if ep_num in (0, 4, 9):
        print(f'  [DIAG ep{ep_num}] Pred stats: mean={ap.mean():.6f} std={ap.std():.6f} min={ap.min():.6f} max={ap.max():.6f}')
        pos_mask = at_bin > 0.5
        if pos_mask.any():
            print(f'    Pred@positive: mean={ap[pos_mask].mean():.6f} std={ap[pos_mask].std():.6f}')
        neg_mask = at_bin < 0.1
        if neg_mask.any():
            print(f'    Pred@negative: mean={ap[neg_mask].mean():.6f} std={ap[neg_mask].std():.6f}')

    v = [i for i in range(at_bin.shape[1]) if 0 < at_bin[:,i].sum() < len(at_bin)]
    if not v: return (0.0, 0)
    try:
        auc = roc_auc_score(at_bin[:,v], ap[:,v], average='macro')
    except Exception as e:
        print(f'  AUC error: {e}')
        auc = 0.0
    return (auc, len(v))

In [ ]:
sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
folds = list(sgkf.split(train_df, train_df['primary_label'].astype(str), train_df['author'].astype(str)))

results = []
for fi in CFG.TRAIN_FOLDS:
    ti, vi = folds[fi]
    print(f'\n{"="*60}')
    print(f'  Fold {fi+1}/{CFG.N_FOLDS} | Train: {len(ti)}, Val: {len(vi)}')
    print(f'{"="*60}')
    tl = DataLoader(BirdDataset(train_df.iloc[ti], True), batch_size=CFG.BATCH_SIZE,
                    shuffle=True, num_workers=CFG.NUM_WORKERS, pin_memory=True,
                    drop_last=True, persistent_workers=True, prefetch_factor=3)
    vl = DataLoader(BirdDataset(train_df.iloc[vi], False), batch_size=CFG.BATCH_SIZE*2,
                    shuffle=False, num_workers=CFG.NUM_WORKERS, pin_memory=True,
                    persistent_workers=True, prefetch_factor=3)

    model = BirdCLEFB0(pretrained=True).to(DEVICE)
    crit = nn.BCEWithLogitsLoss()  # simple BCE, no class weights
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG.EPOCHS, eta_min=1e-5)
    scaler = GradScaler() if USE_AMP else None
    best = 0.0

    for ep in range(CFG.EPOCHS):
        t0 = time.time()
        loss = train_ep(model, tl, crit, opt, scaler, ep_num=ep)
        auc, nc = val(model, vl, ep_num=ep); sched.step()
        tag = ''
        if auc > best:
            best = auc
            torch.save(model.state_dict(), CFG.OUTPUT_DIR/f'best_fold{fi}.pth')
            tag = ' *BEST'
        elapsed = time.time()-t0
        print(f'  Ep {ep+1:02d}/{CFG.EPOCHS} | loss={loss:.4f} auc={auc:.4f} ({nc}sp) | lr={opt.param_groups[0]["lr"]:.6f} | {elapsed:.0f}s{tag}')

    results.append(best)
    print(f'  Fold {fi+1} Best AUC: {best:.4f}')
    del model, opt, sched, scaler, crit; torch.cuda.empty_cache()

print(f'\nCV Mean AUC: {np.mean(results):.4f}')

In [ ]:
import json
meta = {'species_cols': SPECIES_COLS, 'label2idx': LABEL2IDX, 'tax_map': TAX_MAP, 'results': results}
with open(CFG.OUTPUT_DIR / 'train_meta.json', 'w') as f:
    json.dump(meta, f)
for p in sorted(CFG.OUTPUT_DIR.glob('*')):
    print(f'  {p.name} ({p.stat().st_size/1024:.0f}KB)')